# 지륜님 개인 통합 정리용 Notebook 작성본 - T데이터 전체

- 프로젝트명: 성남시 젠트리피케이션 예측/분석
- 담당자명: 지륜
- 담당 파트: 통신 T데이터 통합/전처리/EDA, 공시지가 상승률 EDA
- 작성일: 2026-04-29
- 정리 목적: `T4~T27` 통신 데이터와 공시지가 상승률 EDA의 산출물, 전처리 기준, 검증 결과, 변수 후보를 한 노트북에서 확인할 수 있게 정리한다.
- 최종 산출물 한 줄 요약: `T4~T27` 통신 테이블과 공시지가 2023~2025 상승률 EDA를 함께 정리한다.

> 이 노트북은 새 분석을 처음부터 다시 하는 노트북이 아니라, 지금까지 만든 통합 파일과 EDA 결과를 팀 공유용으로 재정리한 최종 기록용 노트북이다.

> 이 파일은 지륜 전용 템플릿 형식에 맞춘 상세 작성본이다. 공통 양식 요약본은 `개인_통합정리_공통템플릿_작성본_지륜.ipynb`에서 확인한다.


## 1. 작업 개요

### 정리 범위
- 포함: `T4, T5, T6, T7, T8, T9, T10, T11, T12, T13, T14, T16, T20, T21, T22, T23, T24, T25, T26, T27`
- 현재 폴더 기준 미확보: `T15, T17, T18, T19`
- 통합 기간: 2023년 1월부터 2025년 12월까지
- 핵심 흐름: 원본/중간 통합 파일 확인 -> 날짜/지역/코드/결측 점검 -> 최종 parquet/csv 산출물 확정 -> 월별/분기별 EDA -> 모델링 변수 후보 정리

### 기존 작업 노트북 역할
| 노트북 | 역할 | 포함 내용 |
|---|---|---|
| `1차_전처리_small.ipynb` | T4~T12, T14, T16, T20~T24 기초 전처리 | 날짜 컬럼 확인, 범주형 분포, 최종 date/final 파일 저장 |
| `1차_전처리.ipynb` | T13, T25, T26, T27 상세 전처리 | O/D 지역명 결측 보정, 99 코드 유지 판단, CNT/DURATION/코드값 분포 확인 |
| `eda.ipynb` | 월별 인구/유동 지표 EDA | T13 외부유입, T24 유동인구, 경제활동 연령층 proxy, 인구 데이터 결합 |
| `my_eda_월별EDA_백업.ipynb` | 월별 행정동 단위 통합 EDA | 전체 T데이터 구조 검토, 월별 변수 해석 메모, 후보 변수 정리 |
| `my_eda.ipynb` | 분기별 EDA 최종 후보 정리 | T24 행정동 분기 패널, T13/T22/T26 보조 지표, PURPOSE/TRANS_GB 코드 정의 반영 |
| `공시지가_test.ipynb` | 별도 공시지가 EDA | 통신 T데이터 정리 범위 밖이지만 팀 최종 분석에 결합 가능 |


## 2. 최종 입력/산출 파일 정리

아래 표는 현재 `data` 폴더에서 확인한 최종 사용 후보 파일 기준이다. `final_v2`, `date_final`, `final` 파일을 우선 산출물로 보았다.

| 테이블 | 최종 파일명 | 행 수 | 기간 범위 | 단위 | 주요 의미 | 현재 판단 |
|---|---|---:|---|---|---|---|
| T4 | `t4_2023_2025_all_date_final.parquet` | 1,211,106 | 2023-01~2025-12 | 시군구 | 도착지 + 목적 + 성/연령 | 구조 파악/보조 |
| T5 | `t5_2023_2025_all_date_final.parquet` | 43,306,619 | 2023-01-01~2025-12-31 | 행정동 | 도착지 + 목적 + 성/연령 | 구조 파악/보조 |
| T6 | `t6_2023_2025_all_date_final.parquet` | 1,927,203 | 2023-01~2025-12 | 시군구 | 도착지 + 이동수단 + 성/연령 | 구조 파악/보조 |
| T7 | `t7_2023_2025_all_date_final.parquet` | 68,932,750 | 2023-01-01~2025-12-31 | 행정동 | 도착지 + 이동수단 + 성/연령 | 구조 파악/보조 |
| T8 | `t8_2023_2025_all_date_final.parquet` | 1,638,511 | 2023-01~2025-12 | 시군구 | 출발지 + 목적 + 성/연령 | 구조 파악/보조 |
| T9 | `t9_2023_2025_all_date_final.parquet` | 50,298,816 | 2023-01-01~2025-12-31 | 행정동 | 출발지 + 목적 + 성/연령 | 구조 파악/보조 |
| T10 | `t10_2023_2025_all_date_final.parquet` | 1,971,599 | 2023-01~2025-12 | 시군구 | 출발지 + 이동수단 + 성/연령 | 구조 파악/보조 |
| T11 | `t11_2023_2025_all_date_final.parquet` | 71,457,282 | 2023-01-01~2025-12-31 | 행정동 | 출발지 + 이동수단 + 성/연령 | 구조 파악/보조 |
| T12 | `t12_2023_2025_all_final_v2.parquet` | 6,743,756 | 2023-01~2025-12 | 시군구 OD | 출발지->도착지 + 목적 | 구조 파악/보조 |
| T13 | `t13_2023_2025_all_final_v2.parquet` | 279,996,851 | 2023-01-01~2025-12-31 | 행정동 OD | 출발지->도착지 + 목적 | 핵심 후보 |
| T14 | `t14_2023_2025_all_final_v2.parquet` | 8,472,627 | 2023-01~2025-12 | 시군구 OD | 출발지->도착지 + 이동수단 | 구조 파악/보조 |
| T16 | `t16_2023_2025_all_date_final.parquet` | 1,211,106 | 2023-01~2025-12 | 시군구 | 도착지 + 목적 + 체류시간 | 구조 파악/보조 |
| T20 | `t20_2023_2025_all_date_final.csv` | 6,048 | 2023-01~2025-12 | 시군구 | 출발지 + 이동수단 + 거리/탄소 | 참고 |
| T21 | `t21_2023_2025_all_date_final.csv` | 78,840 | 2023-01-01~2025-12-31 | 시군구 | 날짜 + 시간대 | 참고, CNT 없음 |
| T22 | `t22_2023_2025_all_date_final.parquet` | 2,590,644 | 2023-01-01~2025-12-31 | 행정동 | 시간대 + 성/연령 + 내외국인 | 보조 후보 |
| T23 | `t23_2023_2025_all_date_final.parquet` | 4,693,040 | 2023-01-01~2025-12-31 | 시군구 | 시간대 + 목적 유동인구 | 구조 파악/보조 |
| T24 | `t24_2023_2025_all_date_final.parquet` | 43,306,619 | 2023-01-01~2025-12-31 | 행정동 | 시간대 + 목적 유동인구 | 핵심 후보 |
| T25 | `t25_2023_2025_all_final_v2.parquet` | 261,148,533 | 2023-01-01~2025-12-31 | 시군구 OD | 출발지->도착지 + 시간대 + 목적 + 이동수단 | 보조 후보 |
| T26 | `t26_2023_2025_all_final.parquet` | 258,637,442 | 2023-01-01~2025-12-31 | 도착 행정동 | 도착지 + 시간대 + 목적 + 이동수단 + 체류시간 | 핵심 후보 |
| T27 | `t27_2023_2025_all_final.parquet` | 278,322,883 | 2023-01-01~2025-12-31 | 출발 행정동 | 출발지 + 시간대 + 목적 + 이동수단 + 체류시간 | 선택 후보 |

### 현재 폴더 기준 누락 테이블
| 테이블 | 상태 | 메모 |
|---|---|---|
| T15 | 파일 없음 | `data` 폴더에 최종 산출물 없음 |
| T17 | 파일 없음 | `data` 폴더에 최종 산출물 없음 |
| T18 | 파일 없음 | `data` 폴더에 최종 산출물 없음 |
| T19 | 파일 없음 | `data` 폴더에 최종 산출물 없음 |

### 공시지가 EDA 입력 파일
| 파일명 | 행 수 | 컬럼 수 | 기간 | 주요 내용 |
|---|---:|---:|---|---|
| `2차전처리_공시지가데이터.csv` | 15,542 | 13 | 2023~2025 | 필지별 공시지가, 법정동/구, 기준연도/월 |


## 2-1. 원본/중간 파일 -> 최종 산출물 매핑

템플릿의 `원본 파일 -> 최종 parquet 파일 매핑표` 항목을 전체 T데이터 기준으로 확장했다. 현재 `data` 폴더에 남아 있는 파일과 기존 전처리 노트북의 read/write 기록을 함께 기준으로 정리했다.

| 테이블 | 전처리 기록상 입력/중간 파일 | 최종 산출물 | 정리 상태 | 비고 |
|---|---|---|---|---|
| T4 | `t4_all.parquet` | `t4_2023_2025_all_date_final.parquet` | 완료 | 날짜/범주형 확인 후 저장 |
| T5 | `t5_all.parquet` | `t5_2023_2025_all_date_final.parquet` | 완료 | 행정동 도착지 목적 데이터 |
| T6 | `t6_all.parquet` | `t6_2023_2025_all_date_final.parquet` | 완료 | 도착지 이동수단 데이터 |
| T7 | `t7_all.parquet` | `t7_2023_2025_all_date_final.parquet` | 완료 | 행정동 도착지 이동수단 데이터 |
| T8 | `t8_all.parquet` | `t8_2023_2025_all_date_final.parquet` | 완료 | 출발지 목적 데이터 |
| T9 | `t9_all.parquet` | `t9_2023_2025_all_date_final.parquet` | 완료 | 행정동 출발지 목적 데이터 |
| T10 | `t10_all.parquet` | `t10_2023_2025_all_date_final.parquet` | 완료 | 출발지 이동수단 데이터 |
| T11 | `t11_all.parquet` | `t11_2023_2025_all_date_final.parquet` | 완료 | 행정동 출발지 이동수단 데이터 |
| T12 | `t12_2023_2025_all.parquet` -> `t12_2023_2025_all_final.parquet` | `t12_2023_2025_all_final_v2.parquet` | 완료 | 시군구 OD 목적 데이터 |
| T13 | `t13_2023_2025_all.parquet` -> `clean` -> `clean_fixed` -> `final` | `t13_2023_2025_all_final_v2.parquet` | 완료 | O/D 지역명 결측 보정 및 99 코드 유지 |
| T14 | `t14_2023_2025_all.parquet` -> `t14_2023_2025_all_final.parquet` | `t14_2023_2025_all_final_v2.parquet` | 완료 | 시군구 OD 이동수단 데이터 |
| T15 | 없음 | 없음 | 현재 폴더 기준 누락 | `data` 폴더와 기존 노트북 read/write 기록에서 확인 안 됨 |
| T16 | `t16_2023_2025_all.parquet` | `t16_2023_2025_all_date_final.parquet` | 완료 | 목적 + 체류시간 데이터 |
| T17 | 없음 | 없음 | 현재 폴더 기준 누락 | `data` 폴더와 기존 노트북 read/write 기록에서 확인 안 됨 |
| T18 | 없음 | 없음 | 현재 폴더 기준 누락 | `data` 폴더와 기존 노트북 read/write 기록에서 확인 안 됨 |
| T19 | 없음 | 없음 | 현재 폴더 기준 누락 | `data` 폴더와 기존 노트북 read/write 기록에서 확인 안 됨 |
| T20 | `t20_2023_2025_all.csv` | `t20_2023_2025_all_date_final.csv` | 완료 | CSV 유지, 거리/탄소 포함 참고용 |
| T21 | `t21_2023_2025_all.csv` | `t21_2023_2025_all_date_final.csv` | 완료 | CNT 없음, 참고용 |
| T22 | `t22_2023_2025_all.parquet` | `t22_2023_2025_all_date_final.parquet` | 완료 | 성/연령/내외국인 보조 후보 |
| T23 | `t23_2023_2025_all.parquet` | `t23_2023_2025_all_date_final.parquet` | 완료 | 시군구 목적 유동인구 |
| T24 | `t24_2023_2025_all.parquet` | `t24_2023_2025_all_date_final.parquet` | 완료 | 행정동 유동인구 핵심 후보 |
| T25 | `t25_2023_2025_all.parquet` -> `clean` -> `final` | `t25_2023_2025_all_final_v2.parquet` | 완료 | 시군구 OD, O/D 결측 보정 및 99 코드 유지 |
| T26 | `t26_2023_2025_all_clean.parquet` -> `fixed` | `t26_2023_2025_all_final.parquet` | 완료 | 도착 행정동 체류/목적/수단 핵심 후보 |
| T27 | `t27_2023_2025_all.parquet` -> `clean` | `t27_2023_2025_all_final.parquet` | 완료 | 출발 행정동 체류/목적/수단 선택 후보 |

현재 최종 정리본에서 실제 분석 후보로 잡은 파일은 모두 존재한다. 단, T13/T25/T26/T27의 일부 초기 입력/중간 파일은 전처리 노트북 기록에는 남아 있지만 현재 `data` 폴더에는 최종 파일 중심으로 남아 있다.


## 2-2. 누락 여부 재점검 결과

| 점검 항목 | 결과 | 조치 |
|---|---|---|
| `T4~T27` 테이블 포함 여부 | T4~T14, T16, T20~T27 포함. T15/T17/T18/T19는 파일 없음 | 누락 테이블로 명시 |
| 최종 파일 존재 여부 | 정리본의 최종 후보 파일 20개 모두 존재 | 파일 존재 검증 코드 유지 |
| 템플릿의 매핑표 항목 | 첫 정리본에서는 파일 목록과 처리 기준에 흡수되어 있었음 | `2-1. 원본/중간 파일 -> 최종 산출물 매핑` 섹션 추가 |
| 전처리 기준 | 날짜, 지역 매핑, 99 코드, 핵심 컬럼 결측, 코드값 해석 기준 포함 | 유지 |
| EDA 정리 | 월별 EDA, 분기별 EDA, 모델링 후보 변수 포함 | 유지 |
| 통신 외 작업 | 공시지가 EDA는 통신 T데이터 범위 밖 | 작업 노트북 역할표에 별도 표시 |


In [1]:
# 3. 환경 설정
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

PERSON_NAME = "지륜"
PERSON_PART = "통신 T데이터 통합/전처리/EDA"

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "data").exists() else NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
JIRYUN_DIR = PROJECT_ROOT / "Jiryun"

print(f"PROJECT_ROOT: {PROJECT_ROOT.resolve()}")
print(f"DATA_DIR    : {DATA_DIR.resolve()}")
print(f"JIRYUN_DIR  : {JIRYUN_DIR.resolve()}")


PROJECT_ROOT: C:\Users\wlfbs\final_project
DATA_DIR    : C:\Users\wlfbs\final_project\data
JIRYUN_DIR  : C:\Users\wlfbs\final_project\Jiryun


In [2]:
# 4. 전체 T데이터 카탈로그
# file: 현재 data 폴더에서 최종 사용 후보로 보는 파일명
# status: 모델링/팀 공유 시 우선순위 판단
T_CATALOG = {
    "T4":  {"file": "t4_2023_2025_all_date_final.parquet",  "format": "parquet", "unit": "시군구", "meaning": "도착지 + 목적 + 성/연령", "status": "구조 파악/보조"},
    "T5":  {"file": "t5_2023_2025_all_date_final.parquet",  "format": "parquet", "unit": "행정동", "meaning": "도착지 + 목적 + 성/연령", "status": "구조 파악/보조"},
    "T6":  {"file": "t6_2023_2025_all_date_final.parquet",  "format": "parquet", "unit": "시군구", "meaning": "도착지 + 이동수단 + 성/연령", "status": "구조 파악/보조"},
    "T7":  {"file": "t7_2023_2025_all_date_final.parquet",  "format": "parquet", "unit": "행정동", "meaning": "도착지 + 이동수단 + 성/연령", "status": "구조 파악/보조"},
    "T8":  {"file": "t8_2023_2025_all_date_final.parquet",  "format": "parquet", "unit": "시군구", "meaning": "출발지 + 목적 + 성/연령", "status": "구조 파악/보조"},
    "T9":  {"file": "t9_2023_2025_all_date_final.parquet",  "format": "parquet", "unit": "행정동", "meaning": "출발지 + 목적 + 성/연령", "status": "구조 파악/보조"},
    "T10": {"file": "t10_2023_2025_all_date_final.parquet", "format": "parquet", "unit": "시군구", "meaning": "출발지 + 이동수단 + 성/연령", "status": "구조 파악/보조"},
    "T11": {"file": "t11_2023_2025_all_date_final.parquet", "format": "parquet", "unit": "행정동", "meaning": "출발지 + 이동수단 + 성/연령", "status": "구조 파악/보조"},
    "T12": {"file": "t12_2023_2025_all_final_v2.parquet",   "format": "parquet", "unit": "시군구 OD", "meaning": "출발지->도착지 + 목적", "status": "구조 파악/보조"},
    "T13": {"file": "t13_2023_2025_all_final_v2.parquet",   "format": "parquet", "unit": "행정동 OD", "meaning": "출발지->도착지 + 목적", "status": "핵심 후보"},
    "T14": {"file": "t14_2023_2025_all_final_v2.parquet",   "format": "parquet", "unit": "시군구 OD", "meaning": "출발지->도착지 + 이동수단", "status": "구조 파악/보조"},
    "T16": {"file": "t16_2023_2025_all_date_final.parquet", "format": "parquet", "unit": "시군구", "meaning": "도착지 + 목적 + 체류시간", "status": "구조 파악/보조"},
    "T20": {"file": "t20_2023_2025_all_date_final.csv",     "format": "csv", "unit": "시군구", "meaning": "출발지 + 이동수단 + 거리/탄소", "status": "참고"},
    "T21": {"file": "t21_2023_2025_all_date_final.csv",     "format": "csv", "unit": "시군구", "meaning": "날짜 + 시간대", "status": "참고, CNT 없음"},
    "T22": {"file": "t22_2023_2025_all_date_final.parquet", "format": "parquet", "unit": "행정동", "meaning": "시간대 + 성/연령 + 내외국인", "status": "보조 후보"},
    "T23": {"file": "t23_2023_2025_all_date_final.parquet", "format": "parquet", "unit": "시군구", "meaning": "시간대 + 목적 유동인구", "status": "구조 파악/보조"},
    "T24": {"file": "t24_2023_2025_all_date_final.parquet", "format": "parquet", "unit": "행정동", "meaning": "시간대 + 목적 유동인구", "status": "핵심 후보"},
    "T25": {"file": "t25_2023_2025_all_final_v2.parquet",   "format": "parquet", "unit": "시군구 OD", "meaning": "출발지->도착지 + 시간대 + 목적 + 이동수단", "status": "보조 후보"},
    "T26": {"file": "t26_2023_2025_all_final.parquet",      "format": "parquet", "unit": "도착 행정동", "meaning": "도착지 + 시간대 + 목적 + 이동수단 + 체류시간", "status": "핵심 후보"},
    "T27": {"file": "t27_2023_2025_all_final.parquet",      "format": "parquet", "unit": "출발 행정동", "meaning": "출발지 + 시간대 + 목적 + 이동수단 + 체류시간", "status": "선택 후보"},
}

catalog_df = pd.DataFrame.from_dict(T_CATALOG, orient="index").reset_index(names="table")
catalog_df["path"] = catalog_df["file"].apply(lambda x: DATA_DIR / x)
catalog_df["exists"] = catalog_df["path"].apply(lambda p: p.exists())
display(catalog_df[["table", "file", "format", "unit", "meaning", "status", "exists"]])


,table,file,format,unit,meaning,status,exists
0,T4,t4_2023_2025_all_date_final.parquet,parquet,시군구,도착지 + 목적 + 성/연령,구조 파악/보조,True
1,T5,t5_2023_2025_all_date_final.parquet,parquet,행정동,도착지 + 목적 + 성/연령,구조 파악/보조,True
2,T6,t6_2023_2025_all_date_final.parquet,parquet,시군구,도착지 + 이동수단 + 성/연령,구조 파악/보조,True
3,T7,t7_2023_2025_all_date_final.parquet,parquet,행정동,도착지 + 이동수단 + 성/연령,구조 파악/보조,True
4,T8,t8_2023_2025_all_date_final.parquet,parquet,시군구,출발지 + 목적 + 성/연령,구조 파악/보조,True
5,T9,t9_2023_2025_all_date_final.parquet,parquet,행정동,출발지 + 목적 + 성/연령,구조 파악/보조,True
6,T10,t10_2023_2025_all_date_final.parquet,parquet,시군구,출발지 + 이동수단 + 성/연령,구조 파악/보조,True
7,T11,t11_2023_2025_all_date_final.parquet,parquet,행정동,출발지 + 이동수단 + 성/연령,구조 파악/보조,True
8,T12,t12_2023_2025_all_final_v2.parquet,parquet,시군구 OD,출발지->도착지 + 목적,구조 파악/보조,True
9,T13,t13_2023_2025_all_final_v2.parquet,parquet,행정동 OD,출발지->도착지 + 목적,핵심 후보,True


## 3. 파일 존재 여부 및 메타데이터 검증

아래 코드는 대용량 parquet를 전부 읽지 않고, 가능한 한 parquet 메타데이터와 row group statistics로 행 수, 컬럼 수, 기간 범위를 확인한다. CSV는 파일 크기가 작아 직접 확인한다.


In [3]:
DATE_CANDIDATES = ["ETL_YMD", "ETL_YM", "STDR_YYMM", "STD_YM", "base_ym", "date", "month"]
KEY_CANDIDATES = [
    "CNT", "DURATION", "PURPOSE", "TRANS_GB", "SEX_CD", "AGE_GRP",
    "TIME_CD", "D_TIME_CD", "O_TIME_CD", "DOW",
    "ADMI_CD", "ADMI_NM", "CTY_CD", "CTY_NM",
    "D_CTY_NM", "D_ADMI_CD", "D_ADMI_NM", "O_CTY_NM", "O_ADMI_CD", "O_ADMI_NM",
]


def parquet_column_nulls(pf: pq.ParquetFile, column_name: str):
    columns = pf.schema_arrow.names
    if column_name not in columns:
        return None
    col_idx = columns.index(column_name)
    total_nulls = 0
    has_stats = False
    for row_group_idx in range(pf.metadata.num_row_groups):
        stats = pf.metadata.row_group(row_group_idx).column(col_idx).statistics
        if stats is not None:
            total_nulls += stats.null_count or 0
            has_stats = True
    return total_nulls if has_stats else np.nan


def parquet_column_minmax(pf: pq.ParquetFile, column_name: str):
    columns = pf.schema_arrow.names
    if column_name not in columns:
        return (None, None)
    col_idx = columns.index(column_name)
    mins, maxs = [], []
    for row_group_idx in range(pf.metadata.num_row_groups):
        stats = pf.metadata.row_group(row_group_idx).column(col_idx).statistics
        if stats is not None and stats.has_min_max:
            mins.append(stats.min)
            maxs.append(stats.max)
    if not mins:
        return (None, None)
    return (min(mins), max(maxs))


def summarize_file(row):
    table = row["table"]
    path = Path(row["path"])
    result = {
        "table": table,
        "file": path.name,
        "exists": path.exists(),
        "format": row["format"],
        "unit": row["unit"],
        "meaning": row["meaning"],
        "status": row["status"],
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else np.nan,
        "rows": np.nan,
        "columns": np.nan,
        "date_col": None,
        "date_min": None,
        "date_max": None,
        "key_columns": "",
    }
    if not path.exists():
        return result

    if path.suffix.lower() == ".parquet":
        pf = pq.ParquetFile(path)
        cols = pf.schema_arrow.names
        result["rows"] = pf.metadata.num_rows
        result["columns"] = len(cols)
        result["key_columns"] = ", ".join([c for c in KEY_CANDIDATES if c in cols])
        date_col = next((c for c in DATE_CANDIDATES if c in cols), None)
        if date_col:
            result["date_col"] = date_col
            result["date_min"], result["date_max"] = parquet_column_minmax(pf, date_col)
        return result

    if path.suffix.lower() == ".csv":
        header = pd.read_csv(path, nrows=0)
        cols = list(header.columns)
        result["columns"] = len(cols)
        result["rows"] = sum(1 for _ in path.open("rb")) - 1
        result["key_columns"] = ", ".join([c for c in KEY_CANDIDATES if c in cols])
        date_col = next((c for c in DATE_CANDIDATES if c in cols), None)
        if date_col:
            date_series = pd.read_csv(path, usecols=[date_col])[date_col]
            result["date_col"] = date_col
            result["date_min"] = date_series.min()
            result["date_max"] = date_series.max()
        return result

    return result

summary_rows = [summarize_file(row) for _, row in catalog_df.iterrows()]
summary_df = pd.DataFrame(summary_rows)
summary_df["rows"] = summary_df["rows"].astype("Int64")
summary_df["columns"] = summary_df["columns"].astype("Int64")
display(summary_df[["table", "file", "exists", "rows", "columns", "size_mb", "date_col", "date_min", "date_max", "key_columns", "status"]])


,table,file,exists,rows,columns,size_mb,date_col,date_min,date_max,key_columns,status
0,T4,t4_2023_2025_all_date_final.parquet,True,1211106,12,8.38,ETL_YM,2023-01-01 00:00:00,2025-12-01 00:00:00,"CNT, PURPOSE, SEX_CD, AGE_GRP, D_TIME_CD, DOW,...",구조 파악/보조
1,T5,t5_2023_2025_all_date_final.parquet,True,43306619,12,253.33,ETL_YMD,2023-01-01 00:00:00,2025-12-31 00:00:00,"CNT, PURPOSE, SEX_CD, AGE_GRP, D_TIME_CD, D_CT...",구조 파악/보조
2,T6,t6_2023_2025_all_date_final.parquet,True,1927203,12,12.59,ETL_YM,2023-01-01 00:00:00,2025-12-01 00:00:00,"CNT, TRANS_GB, SEX_CD, AGE_GRP, D_TIME_CD, DOW...",구조 파악/보조
3,T7,t7_2023_2025_all_date_final.parquet,True,68932750,12,385.98,ETL_YMD,2023-01-01 00:00:00,2025-12-31 00:00:00,"CNT, TRANS_GB, SEX_CD, AGE_GRP, D_TIME_CD, D_C...",구조 파악/보조
4,T8,t8_2023_2025_all_date_final.parquet,True,1638511,12,10.56,ETL_YM,2023-01-01 00:00:00,2025-12-01 00:00:00,"CNT, PURPOSE, SEX_CD, AGE_GRP, O_TIME_CD, DOW,...",구조 파악/보조
5,T9,t9_2023_2025_all_date_final.parquet,True,50298816,12,289.38,ETL_YMD,2023-01-01 00:00:00,2025-12-31 00:00:00,"CNT, PURPOSE, SEX_CD, AGE_GRP, O_TIME_CD, O_CT...",구조 파악/보조
6,T10,t10_2023_2025_all_date_final.parquet,True,1971599,12,12.93,ETL_YM,2023-01-01 00:00:00,2025-12-01 00:00:00,"CNT, TRANS_GB, SEX_CD, AGE_GRP, O_TIME_CD, DOW...",구조 파악/보조
7,T11,t11_2023_2025_all_date_final.parquet,True,71457282,12,403.37,ETL_YMD,2023-01-01 00:00:00,2025-12-31 00:00:00,"CNT, TRANS_GB, SEX_CD, AGE_GRP, O_TIME_CD, O_C...",구조 파악/보조
8,T12,t12_2023_2025_all_final_v2.parquet,True,6743756,16,42.92,ETL_YM,2023-01-01 00:00:00,2025-12-01 00:00:00,"CNT, PURPOSE, SEX_CD, AGE_GRP, DOW, D_CTY_NM, ...",구조 파악/보조
9,T13,t13_2023_2025_all_final_v2.parquet,True,279996851,17,3537.97,ETL_YMD,2023-01-01,2025-12-31,"CNT, PURPOSE, SEX_CD, AGE_GRP, D_CTY_NM, D_ADM...",핵심 후보


## 4. 통합 및 전처리 기준 문서화

### 공통 처리 기준
- 원본/중간 파일은 직접 수정하지 않고, 분석용 최종 산출물을 `data` 폴더에 별도 저장했다.
- parquet 저장 가능 테이블은 parquet를 최종 후보로 두고, T20/T21처럼 크기가 작거나 기존 산출물이 CSV인 경우 CSV를 유지했다.
- 날짜 컬럼은 `ETL_YM` 또는 `ETL_YMD` 기준으로 2023~2025 전체 기간을 확인했다.
- `CNT`, `DURATION`, `PURPOSE`, `TRANS_GB`, `SEX_CD`, `AGE_GRP`는 임의 삭제하지 않고 분포와 결측 여부를 먼저 확인했다.
- 지역명/O-D 매핑 결측은 코드 기준으로 보정 가능한 값만 보정했고, 데이터 공급처의 미확인 코드로 보이는 `99`는 원본 의미 보존을 위해 유지했다.
- 대용량 테이블은 전체 스캔을 반복하지 않도록 DuckDB와 parquet 메타데이터를 사용해 검증했다.

### 상세 처리 메모
| 구분 | 처리 내용 | 최종 판단 |
|---|---|---|
| 날짜 | `ETL_YM`, `ETL_YMD` 기준 기간 범위 확인 | 2023-01~2025-12 범위 확보 |
| 지역 매핑 | O/D 지역명, 행정동명, 좌표 결측 확인 후 코드 매핑 가능한 값 보정 | 99/미확인 코드는 유지 |
| 핵심 수치 | `CNT`, `DURATION` 분포와 이상치 확인 | 큰 값 자체를 이상치로 단정하지 않고 EDA에서 해석 |
| 코드값 | `PURPOSE`, `TRANS_GB`, `SEX_CD`, `AGE_GRP` 분포 확인 | 코드 정의와 추정값 성격을 별도 기록 |
| 저장 검증 | 최종 parquet/csv 저장 후 재오픈 및 shape 확인 | 최종 후보 파일로 사용 가능 |


In [4]:
# 핵심 컬럼 결측 검증표
# parquet는 통계 메타데이터 기준으로 확인한다. CSV는 직접 읽어서 확인한다.
WATCH_COLUMNS = [
    "CNT", "DURATION", "PURPOSE", "TRANS_GB", "SEX_CD", "AGE_GRP",
    "D_CTY_NM", "D_ADMI_NM", "O_CTY_NM", "O_ADMI_NM", "CTY_NM", "ADMI_NM",
]

null_rows = []
for _, row in catalog_df.iterrows():
    path = Path(row["path"])
    if not path.exists():
        continue
    if path.suffix.lower() == ".parquet":
        pf = pq.ParquetFile(path)
        cols = pf.schema_arrow.names
        for col in WATCH_COLUMNS:
            if col in cols:
                null_rows.append({
                    "table": row["table"],
                    "column": col,
                    "null_count": parquet_column_nulls(pf, col),
                })
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        for col in WATCH_COLUMNS:
            if col in df.columns:
                null_rows.append({
                    "table": row["table"],
                    "column": col,
                    "null_count": int(df[col].isna().sum()),
                })

null_df = pd.DataFrame(null_rows)
if not null_df.empty:
    null_pivot = null_df.pivot(index="table", columns="column", values="null_count").fillna("")
    display(null_pivot)
else:
    print("확인할 핵심 컬럼이 없습니다.")


column,ADMI_NM,AGE_GRP,CNT,CTY_NM,DURATION,D_ADMI_NM,D_CTY_NM,O_ADMI_NM,O_CTY_NM,PURPOSE,SEX_CD,TRANS_GB
table,,,,,,,,,,,,
T10,,0.0,0.0,,,,,,0.0,,0.0,0.0
T11,,0.0,0.0,,,,,0.0,0.0,,0.0,0.0
T12,,0.0,0.0,,,,41.0,,0.0,0.0,0.0,
T13,,0.0,0.0,,,13.0,13.0,8.0,8.0,0.0,0.0,
T14,,0.0,0.0,,,,40.0,,60.0,,0.0,0.0
T16,,0.0,0.0,,0.0,,0.0,,,0.0,0.0,
T20,,,0.0,,,,,,0.0,,,0.0
T21,,,,0.0,,,,,,,,
T22,0.0,,,0.0,,,,,,,,


## 5. 문제 데이터 및 예외 케이스 정리

| 데이터셋 | 컬럼/영역 | 확인 내용 | 처리 방식 | 이유 |
|---|---|---|---|---|
| T12 | `D_CTY_NM` | 일부 지역명 결측 | 최종 파일에 기록, 사용 시 코드/좌표 우선 확인 | 핵심 CNT/코드 컬럼은 결측 없음 |
| T13 | O/D 지역명 | 최종 기준 O측 8건, D측 13건 수준의 지역명 결측 확인 | `99` 미확인 코드 성격으로 유지 | 공급 데이터의 미확인 지역 의미 보존 |
| T14 | O/D 시군구명 | 일부 지역명 결측 | 최종 파일에 기록, 필요 시 코드 기준 보조 | 핵심 CNT/교통수단/성연령 컬럼은 결측 없음 |
| T25 | O/D 시군구명 | O측 4건, D측 11건 수준의 지역명 결측 확인 | `99` 또는 unknown 성격으로 유지 | 임의 삭제 시 이동량 왜곡 가능 |
| T21 | `CNT` 없음 | 시간대/날짜 보조 테이블 성격 | 모델 후보에서 제외, 참고용 유지 | 유동량 직접 집계 변수 없음 |
| T15/T17/T18/T19 | 파일 없음 | 현재 `data` 폴더에 최종 산출물 없음 | 누락 테이블로 표시 | 팀 공유 시 범위 혼선 방지 |

핵심 분석 컬럼인 `CNT`, `DURATION`, `PURPOSE`, `TRANS_GB`, `SEX_CD`, `AGE_GRP`는 최종 후보 파일 기준 결측이 없거나 분석 가능한 상태로 확인했다. 단, 코드값의 의미는 실제 이동 목적/수단이 아니라 통신사가 추정한 값이므로 보조지표로만 사용한다.


## 6. 코드값 정의 및 해석 기준

`PURPOSE`와 `TRANS_GB`는 `my_eda.ipynb`에서 성남시 제공 코드 정의를 반영했다. 다만 통신사가 위치 반경과 이동 패턴으로 추정한 값이므로 실제 목적/실제 이동수단으로 단정하지 않는다.

### PURPOSE 코드
| 코드 | 의미 | 활용 판단 |
|---:|---|---|
| 0 | 귀가 | 보조지표 |
| 1 | 출근 | 보조지표 |
| 2 | 등교 | 보조지표 |
| 3 | 쇼핑 | 상권/방문 성격 보조지표 |
| 4 | 관광 | 방문/외부수요 보조지표 |
| 5 | 병원 | 생활서비스 방문 보조지표 |
| 6 | 기타 | 기타 목적 |

### TRANS_GB 코드
| 코드 | 의미 | 활용 판단 |
|---:|---|---|
| 0 | 차량 | 접근성/교통 보조지표 |
| 1 | 노선버스 | 대중교통 보조지표 |
| 2 | 지하철 | 대중교통 보조지표 |
| 3 | 도보 | 보행 접근성 보조지표 |
| 4 | 고속버스 | 광역 이동 보조지표 |
| 5 | 기차 | 광역 이동 보조지표 |
| 6 | 항공 | 광역 이동 보조지표 |
| 7 | 기타 | 기타 이동수단 |

### 추가 확인 필요 코드
| 컬럼 | 관측/사용 내용 | 주의점 |
|---|---|---|
| `SEX_CD` | `M`, `F`, `W` 값 관측 | `W`의 정확한 의미는 코드북 확인 필요 |
| `AGE_GRP` | 1~12 범위 값 관측 | 연령대 매핑표 확인 후 해석 필요 |


In [5]:
# 코드값 정의표를 데이터프레임으로 관리한다.
PURPOSE_KO = {
    "0": "귀가",
    "1": "출근",
    "2": "등교",
    "3": "쇼핑",
    "4": "관광",
    "5": "병원",
    "6": "기타",
}

TRANS_KO = {
    "0": "차량",
    "1": "노선버스",
    "2": "지하철",
    "3": "도보",
    "4": "고속버스",
    "5": "기차",
    "6": "항공",
    "7": "기타",
}

code_definition = pd.DataFrame(
    [{"column": "PURPOSE", "code": int(k), "meaning": v, "status": "코드 정의 확인 / 값 자체는 통신 추정"} for k, v in PURPOSE_KO.items()]
    + [{"column": "TRANS_GB", "code": int(k), "meaning": v, "status": "코드 정의 확인 / 값 자체는 통신 추정"} for k, v in TRANS_KO.items()]
)

display(code_definition)


,column,code,meaning,status
0,PURPOSE,0,귀가,코드 정의 확인 / 값 자체는 통신 추정
1,PURPOSE,1,출근,코드 정의 확인 / 값 자체는 통신 추정
2,PURPOSE,2,등교,코드 정의 확인 / 값 자체는 통신 추정
3,PURPOSE,3,쇼핑,코드 정의 확인 / 값 자체는 통신 추정
4,PURPOSE,4,관광,코드 정의 확인 / 값 자체는 통신 추정
5,PURPOSE,5,병원,코드 정의 확인 / 값 자체는 통신 추정
6,PURPOSE,6,기타,코드 정의 확인 / 값 자체는 통신 추정
7,TRANS_GB,0,차량,코드 정의 확인 / 값 자체는 통신 추정
8,TRANS_GB,1,노선버스,코드 정의 확인 / 값 자체는 통신 추정
9,TRANS_GB,2,지하철,코드 정의 확인 / 값 자체는 통신 추정


## 7. EDA 정리

### 월별 EDA에서 만든 주요 지표
| 원천 | 지표 | 의미 | 사용 방향 |
|---|---|---|---|
| T24 | `floating_pop` | 행정동별 전체 유동인구 규모 | 기본 규모 변수 |
| T24 | `floating_change_rate` | 유동인구 전월 대비 변화율 | 변화 감지 |
| T24 | `floating_pop_per_1k_pop` | 등록 인구 1,000명당 유동인구 | 행정동 간 규모 보정 |
| T24 | `working_pop` | 경제활동 연령층 유동인구 proxy | 소비/활동 인구 보조 |
| T24 | `working_pop_share` | 전체 유동 중 경제활동 연령층 비중 | 지역 이용자 구성 |
| T13 | `external_inflow` | 성남시 외부에서 해당 행정동으로 들어온 이동량 | 외부 수요/압력 후보 |
| T13 | `external_inflow_change_rate` | 외부유입 전월 대비 변화율 | 외부 유입 변화 감지 |
| 인구 | `TOTAL_POP`, `HOUSEHOLDS` | 등록 인구/세대수 | 통신 지표 규모 보정 및 결합 |

### 분기별 EDA에서 남긴 판단
- 팀 공통 마스터 테이블이 분기 단위라면 통신 데이터도 `행정동-분기` 단위로 집계하는 것이 안정적이다.
- 1차 추천 조합은 `T24 + T13 + T26`이다.
- `T27`은 T26과 중복성을 확인한 뒤 선택적으로 추가한다.
- `T25`는 시군구 OD라 행정동 모델에서는 해상도가 낮아 보조 변수로 둔다.
- `PURPOSE`, `TRANS_GB` 기반 변수는 추정 목적/추정 이동수단이므로 모델의 보조 설명변수로만 사용한다.


In [6]:
# 선택 실행용 EDA 쿼리 예시
# 대용량 전체 스캔이 발생할 수 있으므로 기본값은 False로 둔다.
RUN_HEAVY_EDA = False

con = duckdb.connect()


def scan_sql(table):
    info = T_CATALOG[table]
    path = (DATA_DIR / info["file"]).as_posix()
    if info["format"] == "csv":
        return f"read_csv_auto('{path}', header=true)"
    return f"read_parquet('{path}')"


def q(sql):
    return con.execute(sql).fetchdf()

if RUN_HEAVY_EDA:
    # T24 행정동-분기 유동인구 규모 예시
    t24_quarterly = q(f'''
        SELECT
            DATE_TRUNC('quarter', CAST(ETL_YMD AS DATE)) AS base_quarter,
            CAST(ADMI_CD AS BIGINT) AS ADMI_CD,
            CTY_NM,
            ADMI_NM,
            SUM(CNT) AS floating_pop_sum,
            SUM(CASE WHEN TRY_CAST(TIME_CD AS INTEGER) BETWEEN 0 AND 5 THEN CNT ELSE 0 END) AS night_cnt,
            SUM(CASE WHEN TRY_CAST(TIME_CD AS INTEGER) BETWEEN 11 AND 14 THEN CNT ELSE 0 END) AS lunch_cnt,
            SUM(CASE WHEN TRY_CAST(TIME_CD AS INTEGER) BETWEEN 17 AND 21 THEN CNT ELSE 0 END) AS evening_cnt
        FROM {scan_sql('T24')}
        GROUP BY 1, 2, 3, 4
        ORDER BY 1, 2
    ''')
    t24_quarterly["night_ratio"] = t24_quarterly["night_cnt"] / t24_quarterly["floating_pop_sum"]
    display(t24_quarterly.head())
else:
    print("대용량 EDA 쿼리는 RUN_HEAVY_EDA=True로 바꾼 뒤 필요한 셀만 실행하세요.")


대용량 EDA 쿼리는 RUN_HEAVY_EDA=True로 바꾼 뒤 필요한 셀만 실행하세요.


## 7-1. 통계 검증 상태

현재 정리본에서 검증된 것과 아직 검증되지 않은 것을 분리한다. 여기서 말하는 검증은 두 종류가 다르다.

| 구분 | 현재 상태 | 의미 |
|---|---|---|
| 데이터 품질 검증 | 완료 | 파일 존재, 행 수, 컬럼 수, 기간 범위, 핵심 컬럼 결측 여부 확인 |
| 전처리 검증 | 완료 | 최종 산출물 재오픈, 지역명 결측/99 코드 처리 기준 문서화 |
| EDA 수준 검토 | 일부 완료 | 월별/분기별 추세, 상관, 변동성, 후보 변수 방향성 확인 |
| 통계적 유의성 검정 | 일부만 완료 | 기존 `eda.ipynb`에서 일부 지표의 변동성, 지역 차이, 상관, 추세를 점검했지만 전체 T4~T27 변수에 대해 모두 검정한 것은 아님 |
| 기존 EDA 통계 점검 | 일부 완료 | 변동성 요약, 지역 차이 검정, 상관계수, 월별 방향성 확인을 수행했으나 최종 feature table 검정은 별도 필요 |
| 모델링 검증 | 미완료 | 최종 목표변수와 마스터 테이블이 확정된 뒤 train/test, 변수 중요도, 성능 검증 필요 |

### 현재 문서에서 조심해야 할 표현
- `핵심 후보`, `보조 후보`는 통계적으로 유의하다는 뜻이 아니라 **EDA와 데이터 구조상 먼저 써볼 후보**라는 뜻이다.
- `PURPOSE`, `TRANS_GB`는 코드 정의는 확인했지만 통신 추정값이므로 실제 목적/수단으로 단정하지 않는다.
- 최종 보고서에는 “통계적으로 검증됨”이 아니라 “EDA 기반 후보로 선정했고, 최종 모델링 단계에서 검증 예정”이라고 쓰는 것이 안전하다.

### 추가로 돌려야 할 통계 검증
| 검증 | 목적 | 적용 대상 |
|---|---|---|
| 기술통계/분포 | 변수의 결측, 왜도, 극단값 확인 | 모든 최종 feature |
| 지역 차이 검정 | 구/행정동별 차이가 우연인지 확인 | T24 유동량, T13 외부유입, T26 체류시간 |
| 상관/중복성 확인 | 비슷한 변수가 너무 많이 들어가는지 확인 | 최종 feature 전체 |
| 추세 검정 | 시간에 따라 증가/감소 경향이 있는지 확인 | 월별/분기별 feature |
| 목표변수 연관성 | 젠트리피케이션 proxy/target과 실제 관련 있는지 확인 | 최종 모델링 테이블 |


In [7]:
# 통계 검증 상태표와 재사용 함수
# 이 셀은 대용량 원본을 바로 스캔하지 않는다.
# 최종 feature table을 만든 뒤 make_stat_validation_summary()에 넣어 검증한다.
try:
    from scipy import stats
except ImportError:
    stats = None

stat_validation_status = pd.DataFrame([
    {"check": "파일/기간/행수/컬럼 검증", "status": "완료", "evidence": "summary_df, catalog_df"},
    {"check": "핵심 컬럼 결측 검증", "status": "완료", "evidence": "null_df/null_pivot"},
    {"check": "전처리 기준 문서화", "status": "완료", "evidence": "지역 매핑, 99 코드 유지, 코드값 해석 기준"},
    {"check": "월별/분기별 EDA", "status": "일부 완료", "evidence": "eda.ipynb, my_eda.ipynb"},
    {"check": "전체 feature 통계적 유의성 검정", "status": "미완료", "evidence": "최종 feature table 확정 후 실행 필요"},
    {"check": "목표변수와의 모델링 검증", "status": "미완료", "evidence": "팀 공통 target/master table 확정 후 실행 필요"},
])
display(stat_validation_status)


def coefficient_of_variation(series: pd.Series):
    clean = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if clean.empty or clean.mean() == 0:
        return np.nan
    return clean.std() / abs(clean.mean())


def make_stat_validation_summary(df: pd.DataFrame, feature_cols, group_col=None, time_col=None):
    "최종 feature table에 대해 기술통계, 상관, 지역 차이, 추세 검정을 한 번에 확인한다."
    result = {}
    features = [c for c in feature_cols if c in df.columns]
    if not features:
        raise ValueError("feature_cols 중 df에 존재하는 컬럼이 없습니다.")

    numeric_df = df[features].apply(pd.to_numeric, errors="coerce")
    distribution = numeric_df.agg(["count", "mean", "std", "min", "median", "max"]).T
    distribution["missing"] = numeric_df.isna().sum()
    distribution["missing_rate"] = numeric_df.isna().mean()
    distribution["cv"] = [coefficient_of_variation(numeric_df[c]) for c in features]
    distribution["iqr"] = numeric_df.quantile(0.75) - numeric_df.quantile(0.25)
    result["distribution"] = distribution

    if len(features) >= 2:
        result["spearman_corr"] = numeric_df.corr(method="spearman")

    if group_col and group_col in df.columns and stats is not None:
        group_tests = []
        for col in features:
            groups = [
                pd.to_numeric(part[col], errors="coerce").dropna()
                for _, part in df[[group_col, col]].dropna(subset=[group_col]).groupby(group_col)
            ]
            groups = [g for g in groups if len(g) >= 5]
            if len(groups) >= 2:
                stat, p_value = stats.kruskal(*groups)
                group_tests.append({"feature": col, "test": "Kruskal-Wallis", "p_value": p_value})
        result["group_difference_tests"] = pd.DataFrame(group_tests)

    if time_col and time_col in df.columns and stats is not None:
        trend_tests = []
        time_rank = pd.to_datetime(df[time_col], errors="coerce").rank(method="dense")
        for col in features:
            temp = pd.DataFrame({"time_rank": time_rank, "value": numeric_df[col]}).dropna()
            if len(temp) >= 10 and temp["value"].nunique() > 1:
                corr, p_value = stats.spearmanr(temp["time_rank"], temp["value"])
                trend_tests.append({"feature": col, "test": "Spearman time trend", "rho": corr, "p_value": p_value})
        result["time_trend_tests"] = pd.DataFrame(trend_tests)

    return result

print("최종 feature table이 만들어지면 예: make_stat_validation_summary(feature_df, feature_cols, group_col='CTY_NM', time_col='base_quarter')")


,check,status,evidence
0,파일/기간/행수/컬럼 검증,완료,"summary_df, catalog_df"
1,핵심 컬럼 결측 검증,완료,null_df/null_pivot
2,전처리 기준 문서화,완료,"지역 매핑, 99 코드 유지, 코드값 해석 기준"
3,월별/분기별 EDA,일부 완료,"eda.ipynb, my_eda.ipynb"
4,전체 feature 통계적 유의성 검정,미완료,최종 feature table 확정 후 실행 필요
5,목표변수와의 모델링 검증,미완료,팀 공통 target/master table 확정 후 실행 필요


최종 feature table이 만들어지면 예: make_stat_validation_summary(feature_df, feature_cols, group_col='CTY_NM', time_col='base_quarter')


## 7-2. 공시지가 EDA 정리

통신 T데이터 외에 `공시지가_test.ipynb`에서 공시지가 상승률 EDA도 진행했다. 이 내용은 모델링에서 지역 토지가치 변화 보조지표로 연결할 수 있다.

| 항목 | 내용 |
|---|---|
| 입력 파일 | `data/2차전처리_공시지가데이터.csv` |
| 작업 노트북 | `Jiryun/공시지가_test.ipynb` |
| 데이터 규모 | 15,542행, 13컬럼 |
| 기간 | 2023~2025년, 1월 기준 중심. 7월 자료는 소수 존재 |
| 공간 단위 | 법정동/구, 필지 고유번호 기준 |
| 주요 컬럼 | `고유번호`, `법정동코드`, `법정동명`, `구`, `기준연도`, `기준월`, `공시지가` |

### 계산한 지표
| 지표 | 계산 방식 | 의미 |
|---|---|---|
| `공시지가_2023`, `공시지가_2024`, `공시지가_2025` | 같은 `고유번호`의 1월 공시지가를 연도별 wide 형태로 변환 | 연도별 가격 수준 |
| `상승률_23_24` | `공시지가_2024 / 공시지가_2023 - 1` | 2023~2024 상승률 |
| `상승률_24_25` | `공시지가_2025 / 공시지가_2024 - 1` | 2024~2025 상승률 |
| `상승률_23_25` | `공시지가_2025 / 공시지가_2023 - 1` | 2023~2025 누적 상승률 |
| `연평균상승률_23_25` | `(공시지가_2025 / 공시지가_2023) ** (1/2) - 1` | 2년 연평균 상승률 |

### EDA 결과 메모
- 1월 기준 전체 고유번호 5,178개 중 2023~2025가 모두 존재하는 필지는 5,176개다.
- 법정동은 43개이며, `필지수 >= 20` 조건을 적용하면 40개 법정동이 비교 대상이다.
- 2023~2025 누적 상승률 중앙값 상위는 수정구 시흥동, 분당구 삼평동, 분당구 백현동, 수정구 금토동, 분당구 서현동 순으로 확인했다.
- 하위는 수정구 상적동, 중원구 상대원동, 중원구 금광동, 중원구 여수동, 중원구 도촌동 순으로 확인했다.
- 법정동 대표값은 평균보다 중앙값을 우선 사용했다. 일부 필지의 극단값이 평균을 크게 흔들 수 있기 때문이다.
- 공시지가 상승률은 분기별 통신 지표처럼 직접 해석하기보다, 연도별 지역 토지가치 변화 보조지표로 쓰는 것이 안전하다.
- 통신 데이터는 행정동 기준이 많고 공시지가는 법정동 기준이므로, 최종 결합 전 법정동-행정동 매핑 기준을 따로 검증해야 한다.


## 7-3. 보조 작업 간단 메모

아래 내용은 주요 결론이 아니라 기존 노트북을 다시 보며 빠뜨리지 않도록 간단히 남긴 보조 메모다.

| 항목 | 간단 메모 |
|---|---|
| T20 | `DISTANCE`, `CARBON_EMISSIONS`가 있어 거리/탄소 참고 변수로만 기록한다. |
| T21 | `CNT`가 없으므로 직접 feature 후보보다는 시간대 참고용으로 둔다. |
| T11 | 작업 중 중복 확인 과정이 있었지만 최종 파일 기준으로 정리되었으므로 중요 이슈로 따로 해석하지 않는다. |
| T22 | 생산가능/소비가능 연령층 비중은 보조 후보로만 기록한다. |
| 통계 점검 | 변동성, 지역 차이, 상관, 월별 방향성은 일부 EDA에서 봤지만 전체 feature 검증은 아직 미완료다. |


## 8. 모델링용 테이블 우선순위

| 우선순위 | 테이블 | 사용 판단 | 이유 |
|---:|---|---|---|
| 1 | T24 | 기본 feature 후보 | 행정동 단위 유동인구라 `base_quarter + ADMI_CD`로 바로 집계하기 좋음 |
| 2 | T13 | 핵심 추가 후보 | 행정동 OD라 외부유입, 유출입 구조, 이동량 증가를 만들기 좋음 |
| 3 | T26 | 핵심 추가 후보 | 도착 행정동 기준 체류시간과 목적/수단 특성을 만들기 좋음 |
| 4 | T27 | 선택 추가 후보 | 출발 행정동 기준 이동수단/목적/체류 특성을 보완하되 T26과 중복 확인 필요 |
| 보조 | T25 | 참고 후보 | 시군구 OD라 행정동 모델에서는 해상도가 낮아 보조 지표에 적합 |
| 보조 | T22 | 참고 후보 | 행정동 성·연령·내외국인 구성 보조 지표 가능 |
| 보류 | T4~T12, T14, T16, T20, T21, T23 | 구조 파악/필요 시 추가 | 핵심 후보와 중복되거나 모델 단위와 맞추기 전 추가 검토 필요 |

1차 모델링은 `T24 + T13 + T26`으로 시작하고, 변수 중요도와 중복성 확인 후 `T27`, `T25`, `T22`를 추가 검토한다.


## 9. 최종 변수 연결표

| 원천 | 파생변수 후보 | 집계 기준 | 의미 | 사용 방향 |
|---|---|---|---|---|
| T24 | `q_floating_pop_sum` | `base_quarter + ADMI_CD` | 분기별 행정동 전체 유동량 | 기본 규모 변수 |
| T24 | `q_floating_pop_per_1k_pop` | `base_quarter + ADMI_CD` | 인구 1,000명당 유동량 | 규모 보정 변수 |
| T24 | `q_purpose_*_ratio` | 목적별 CNT / 전체 CNT | 목적별 유동 비중 | 지역 성격 비교 |
| T24 | `q_night_ratio` | 야간 CNT / 전체 CNT | 야간 유동 비중 | 주거/상권 혼합 특성 |
| T24 | `q_lunch_ratio`, `q_evening_ratio` | 시간대별 CNT / 전체 CNT | 점심/저녁 활동 비중 | 상권 시간대 특성 |
| T24 | `q_age_20_40_ratio` | 연령대 CNT / 전체 CNT | 주요 소비/경제활동 연령층 비중 | 소비층 proxy |
| T13 | `q_inflow_cnt` | 도착 행정동 기준 | 해당 행정동으로 들어온 이동량 | 유입 규모 |
| T13 | `q_outflow_cnt` | 출발 행정동 기준 | 해당 행정동에서 나간 이동량 | 유출 규모 |
| T13 | `q_external_inflow_cnt` | 외부 지역 -> 해당 행정동 | 외부유입량 | 외부 수요 증가 proxy |
| T13 | `q_external_inflow_ratio` | 외부유입 / 전체유입 | 외부유입 비중 | 젠트리피케이션 압력 후보 |
| T26 | `q_avg_stay_time` | 도착 행정동 기준 | 평균 체류시간 | 체류형 상권 여부 |
| T26 | `q_long_stay_ratio` | 장기체류 CNT / 전체 CNT | 장기체류 비중 | 체류 강도 |
| T26 | `q_purpose_stay_avg_*` | PURPOSE별 DURATION 평균 | 목적별 체류시간 | 방문 목적별 체류 특성 |
| T27 | `q_transport_*_ratio` | TRANS_GB별 CNT 비중 | 이동수단별 비중 | 접근성/교통 특성 |
| T27 | `q_purpose_transport_*` | PURPOSE + TRANS_GB | 목적·수단 조합 | 접근 방식과 방문 목적 결합 |
| T25 | `q_city_inflow_pressure` | 시군구 OD 기준 | 구 단위 유입 압력 | 행정동 모델 보조 변수 |
| T22 | `q_age_sex_foreigner_mix` | 행정동 + 시간대 | 성·연령·내외국인 구성 | 인구 구성 보조 변수 |
| T22 | `working_age_share`, `consumer_age_share` | 행정동 + 시간대 | 생산가능/소비가능 연령층 비중 | 인구 구성 보조 변수 |
| T20 | `distance_sum`, `distance_mean` | 시군구 + 월/요일 | 이동 거리 규모/평균 | 접근성 보조 변수 |
| T20 | `carbon_emissions_sum`, `carbon_emissions_mean` | 시군구 + 월/요일 | 탄소배출량 규모/평균 | 이동수단/거리 파생 보조 |
| 공시지가 | `official_land_price_2023/2024/2025` | 고유번호 + 기준연도 | 연도별 필지 공시지가 | 가격 수준 보조 |
| 공시지가 | `official_land_price_growth_23_25` | 고유번호 | 2023~2025 누적 상승률 | 토지가치 상승 속도 |
| 공시지가 | `dong_land_price_growth_median` | 법정동 | 법정동 상승률 중앙값 | 지역 단위 보조 feature |

### 공통 원본 컬럼 연결
| 원본 컬럼 | 표준 의미 | 사용 방향 |
|---|---|---|
| `ETL_YM`, `ETL_YMD` | 기준 월/일 | 월별 또는 분기별 기준 생성 |
| `CNT` | 집계 건수/유동량 | 대부분의 핵심 feature 원천 |
| `DURATION` | 체류시간 | T16/T26/T27 체류 변수 원천 |
| `PURPOSE` | 추정 이동 목적 코드 | 목적별 비중/목적별 체류 보조 |
| `TRANS_GB` | 추정 이동수단 코드 | 교통수단별 비중 보조 |
| `SEX_CD`, `AGE_GRP` | 성별/연령대 코드 | 이용자 구성 보조 |
| `TIME_CD`, `D_TIME_CD`, `O_TIME_CD` | 시간대 코드 | 야간/점심/저녁 시간대 지표 |
| `ADMI_CD`, `D_ADMI_CD`, `O_ADMI_CD` | 행정동 코드 | 행정동 단위 집계 키 |
| `CTY_NM`, `ADMI_NM`, `D_*`, `O_*` | 지역명 | 해석/시각화 라벨 |


## 10. 최종 산출물 요약표

| 산출물 | 파일/노트북 | 내용 | 팀 공유 시 사용 |
|---|---|---|---|
| T데이터 최종 파일 목록 | `data/t*_2023_2025_all_*` | T4~T27 중 확보된 통신 데이터 최종 parquet/csv | 데이터 목록 공유 |
| 전처리 기록 | `1차_전처리_small.ipynb`, `1차_전처리.ipynb` | 날짜, 결측, 지역 매핑, 코드값, 저장 검증 | 처리 근거 확인 |
| 월별 EDA | `eda.ipynb`, `my_eda_월별EDA_백업.ipynb` | 유동인구/외부유입/인구 결합/월별 지표 해석 | 월별 변수 후보 확인 |
| 분기별 EDA | `my_eda.ipynb` | 행정동-분기 패널, T24/T13/T26 중심 변수 후보 | 모델링 단위 확정 전 검토 |
| 공시지가 EDA | `공시지가_test.ipynb` | 필지별/법정동별/구별 공시지가 상승률, 시각화, 해석 메모 | 토지가치 변화 보조지표 |
| 최종 정리본 | 현재 노트북 | 전체 T데이터 파일, 공시지가 EDA, 전처리 기준, EDA 판단, 변수 연결표 통합 | 튜터님/팀 공유용 |


## 11. 최종 체크리스트

- [x] `T4~T27` 중 현재 확보된 최종 산출물 목록 정리
- [x] `T15/T17/T18/T19` 현재 폴더 기준 누락 표시
- [x] 테이블별 행 수, 컬럼 수, 기간 범위 정리
- [x] 원본/중간 파일 -> 최종 산출물 매핑표 보완
- [x] 통합/전처리 기준 문서화
- [x] 핵심 컬럼 결측 검증 코드 정리
- [x] 문제 데이터 및 예외 케이스 정리
- [x] `PURPOSE`, `TRANS_GB` 코드 정의와 해석 주의점 정리
- [x] 월별/분기별 EDA 결과 기반 변수 후보 정리
- [x] 통계 검증 완료/미완료 상태 구분
- [x] 공시지가 EDA 정리 포함 완료
- [x] 기존 노트북 재대조 후 세부 누락 보완 완료
- [x] 모델링용 테이블 우선순위 정리
- [x] 최종 변수 연결표 작성

## 12. 팀 공유용 5줄 요약

1. 통신 T데이터는 현재 `T4~T27` 중 확보된 20개 테이블을 2023~2025 최종 parquet/csv 산출물 기준으로 정리했고, 공시지가 상승률 EDA도 함께 포함했다.
2. 전처리에서는 날짜 범위, 지역 매핑, 핵심 컬럼 결측, 코드값 분포를 확인했고, 공급 데이터의 미확인 코드로 보이는 `99`는 원본 의미 보존을 위해 유지했다.
3. 월별 EDA에서는 T24 유동인구와 T13 외부유입을 중심으로 행정동별 변화율과 인구 보정 지표를 만들었다.
4. 분기별 모델링 후보는 `T24 + T13 + T26`을 1차 조합으로 보되, 아직 통계적 유의성이 확정된 것은 아니다.
5. 최종 목표변수와 마스터 테이블이 확정되면 지역 차이, 상관/중복성, 시간 추세, 모델 성능, 법정동-행정동 매핑 검증을 추가로 진행해야 한다.
